<a href="https://colab.research.google.com/github/kej534923-maker/ECON5200-Applied-Data-Analytics/blob/main/verification_log_md.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 1: Diagnose — STL Decomposition Bug

### What was wrong

STL was applied directly to the raw retail sales series (`RSXFSN`), which assumes an **additive decomposition**. However, the data exhibits **multiplicative seasonality**, where seasonal fluctuations grow with the level of the series.

---

### Why this happens

In multiplicative processes, the seasonal component scales with the trend:

\[
y_t = T_t \times S_t \times E_t
\]

As the series grows over time, the absolute magnitude of seasonal fluctuations also increases. Applying additive STL:

\[
y_t = T_t + S_t + E_t
\]

forces STL to treat seasonal effects as constant, which leads to an apparent **increase in seasonal amplitude over time**.

---

### Fix applied

Log-transform the series before applying STL:

```python
log_series = np.log(series)
stl = STL(log_series, period=12, robust=True)
res = stl.fit()

## Part 2: Diagnose — Flawed ADF Test

### What was wrong

The ADF test was run with `regression='n'` (no constant, no trend), which is inappropriate for GDP data.

### Why this happens

Real GDP has both a non-zero mean and a clear upward deterministic trend.  
Using `regression='n'` omits these deterministic components and misspecifies the ADF regression. This can bias the test statistic and falsely suggest stationarity.

### Fix applied

Changed the ADF specification to `regression='ct'` and added a KPSS test with `regression='ct'`.

### Verified output

- ADF p-value with `regression='ct'` > 0.05  
- KPSS p-value with `regression='ct'` < 0.05  

**2×2 verdict:** NON-STATIONARY ✓

### Concept violated

Proper specification of deterministic components in unit root testing.  
ADF tests must include the appropriate constant and trend terms when the series has a deterministic trend.

## Part 3: EXTEND — MSTL for Multiple Seasonal Periods

### 1. Does MSTL successfully separate the daily and weekly cycles?

Yes. MSTL clearly separates the two seasonal components:

- The **daily cycle (24h)** shows high-frequency oscillations repeating every day.
- The **weekly cycle (168h)** shows a slower-moving pattern across days.

These two components are visually distinct in the decomposition plot, with different frequencies and amplitudes, indicating that MSTL successfully captures both seasonalities.

---

### 2. Is the residual standard deviation close to the true noise level (15 MW)? What does this tell you?

The residual standard deviation is approximately **12.24 MW**, which is close to the true noise level of **15 MW**.

This suggests that MSTL has effectively extracted most of the signal (trend + seasonal components), leaving primarily random noise in the residuals. The relatively small difference indicates a **high-quality decomposition**.

---

### 3. How would you add an annual seasonal cycle (period = 8760)?

You can include an additional seasonal period in the MSTL call:

```python
from statsmodels.tsa.seasonal import MSTL

mstl = MSTL(demand_series, periods=[24, 168, 8760])
res = mstl.fit()

## Part 4: EXTEND — Block Bootstrap for Trend Uncertainty

### 1. Is the confidence band wider around recessions (2008, 2020) or expansions? Why?

The confidence band is noticeably **wider during recession periods** (e.g., 2008 and 2020) compared to expansion periods.

This occurs because economic shocks during recessions increase volatility in the residuals. When these higher-variance residual blocks are resampled in the bootstrap procedure, they generate a wider distribution of possible trend estimates, leading to **larger uncertainty bands**.

---

### 2. Why do we use block bootstrap instead of standard i.i.d. bootstrap?

We use **block bootstrap** because the residuals from time series models exhibit **autocorrelation**.

Standard i.i.d. bootstrap assumes observations are independent. If we shuffle residuals independently:

- We destroy the temporal dependence structure
- We underestimate variability in persistent shocks
- The resulting confidence intervals become **too narrow and misleading**

Block bootstrap preserves local time dependence by resampling contiguous blocks, making it appropriate for time series data.

---

### 3. How does the choice of block_size=8 affect the results?

A block size of 8 preserves short-term autocorrelation in the residuals while still allowing enough variation across bootstrap samples.

- If `block_size = 1` (i.i.d. bootstrap):
  - Autocorrelation is destroyed
  - Confidence bands become **too narrow**

- If `block_size = 20`:
  - Stronger dependence is preserved
  - Fewer independent resampled blocks
  - Confidence bands become **wider but less stable**

Thus, `block_size = 8` provides a **balance between bias and variance** in the bootstrap estimates.

---

### Concept demonstrated

**Time series bootstrap under dependence**

Block bootstrap accounts for autocorrelation in residuals, providing more realistic uncertainty estimates than i.i.d. resampling.